In [1]:
import os
# os.environ['CUDA_VISIBLE_DEVICES'] = '0,1'
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [2]:
import lm_eval
from lm_eval.tasks import TaskManager
from lm_eval.evaluator import simple_evaluate
from lm_eval.utils import make_table

# Constants

In [3]:
TAWJEEH_DATASET_NAME = 'AraSum'
HF_EXPERIMENTAL_DATASET_NAME = 'KFUPM-JRCAI/AraSum_arabic_experimental'
TASK_NAME='summarization'
MODEL_PATH = "/raid_storage/shared_models/Qwen3-8B-Base"
MODEL_NAME = "Qwen3-8B"
TUNED_MODEL_PATH = None
BATCH_SIZE = 40

In [4]:
TOKENIZER_PATH = MODEL_PATH

# Building the prompts dataset

In [5]:
import requests
 
from tqdm.auto import tqdm
 
prompts = None
 
tries = 10
for i in tqdm(range(tries)):
    api_response = requests.get(url='https://promptlab.up.railway.app/api/prompt/list?project_secret_key=6Wirj')
    if api_response.ok:
        prompts = api_response.json()
        break
if not prompts:
    raise Exception('Failed to fetch prompts')
prompts[:5]

  0%|          | 0/10 [00:00<?, ?it/s]

[{'id': 14901,
  'tags': [],
  'name': 'A Simple Test Prompt',
  'task': {'name': 'dialect identification'},
  'status': 'DRAFT',
  'template': 'Please predict the most suitable dialect for the following text: {{arabic}}\xa0\r\n|||{{answer_choices[label]}}',
  'created_by': 'irfan',
  'dataset_name': 'arbml/AraBench_dev',
  'dataset_subset': 'default',
  'answer_choices': ['Tunisian',
   'MSA',
   'Morrocan',
   'Qatari',
   'Egyptian',
   'Lebanese'],
  'text_direction': 'ltr'},
 {'id': 14898,
  'tags': ['', 'Zero-shot COT'],
  'name': 'Prompt with zero-shot chain of thoughts',
  'task': {'name': 'claim verification'},
  'status': 'APPROVED',
  'template': "For the following task you have to label if the two sentences are of on of the following labels: {% for choice in answer_choices %}{{ choice }}{% if not loop.last %} or {% endif %}{% endfor %}. Sentence 1: {{s1}}\xa0 and sentence 2: {{s2}}.\r\nLet's think step by step:\r\n|||\r\n{{answer_choices[label]}}",
  'created_by': 'ahmed',


In [6]:
filtered_prompts = list(filter(lambda prompt: prompt['status'] == 'APPROVED' and prompt['text_direction'].lower() == 'ltr', prompts))
len(filtered_prompts)

352

### Get the dataset prompts

In [7]:
# you can either filter by task or dataset
dataset_prompts = list(
    filter(
        lambda prompt: TAWJEEH_DATASET_NAME in prompt['dataset_name'],
        filtered_prompts,
    )
)
len(dataset_prompts)

6

In [8]:
SELECTED_PROMPTS_IDS = [
    14891,
    14857,
    14736,
    14735,
    14628,
]

In [9]:
dataset_prompts = list(filter(lambda prompt: prompt['id'] in SELECTED_PROMPTS_IDS, filtered_prompts))
len(dataset_prompts)

5

### Download the dataset

In [10]:
import datasets

In [11]:
hf_exp_dataset = datasets.load_dataset(HF_EXPERIMENTAL_DATASET_NAME)
hf_exp_dataset

DatasetDict({
    test: Dataset({
        features: ['index', 'summary', 'article'],
        num_rows: 10000
    })
})

### Merge the prompts

In [12]:
from jinja2 import Environment, StrictUndefined

In [13]:
def apply_template(prompt_template, sample):
    try:
        template = prompt_template['template']
        env = Environment(undefined=StrictUndefined)
        if "|||" not in template:
            raise ValueError("No ||| dividor")
        template = env.from_string(template)
        rendered_template = template.render(**sample)
        return rendered_template
    except Exception as e:
        print(prompt_template)
        print(sample)
        raise e

Perform generation on one example prompt, for experimentation

In [14]:
example_prompt_template = dataset_prompts[0]
print(apply_template(example_prompt_template, hf_exp_dataset['test'][1]))

You are an expert Arabic text summarizer. The following article:
"حذر زعيم الخضر الألماني جيم أوزدمير (معارضة) من منح تركيا حق استضافة بطولة كأس أوروبا لكرة القدم لعام 2024، مشيراً إلى وجود ""خطر كبير من استغلال نظام أردوغان (لهذا الحدث) لأغراض سياسية"". وذكر السياسي الألماني في حوار مع مجلة ""شبورتب يلد"" الرياضية الألمانية اليوم الأربعاء (20 سبتمبر/ أيلول 2017)، أنه ""قلق جداً من أن تقوم دولة تحتجز رعايا ألمان كرهائن وتضعهم في السجن، لسبب وحيد يكمن في أنهم يمارسون عملهم كصحفيين أو كناشطين في مجال حقوق الانسان""، بتنظيم تظاهرة رياضية، فـ""كرة القدم لا يجب أن تصبح أداة في يد ديكتاتوريين ومستبدين"". وتأتي هذه التصريحات في أوج الحملة الانتخابية ياخذ ملف تركيا فيها حيزاً كبيراً بين المتنافسين. ومن المنتظر أن يحدد الاتحاد الأوروبي لكرة القدم (يويفا) في سبتمبر/ أيلول من العام القادم هوية البلد المنظم لكأس أوروبا 2024، مع العلم أن ألمانيا وتركيا مرشحتان لنيل شرف تنظيم البطولة.    و.ب/ ع.غ (DW)" 
can be summarized as:
|||
"زعيم حزب الخضر الألماني المنحدر من أصول تركية والمعروف بمواقفه المناوئ

merge prompts

In [15]:
for prompt in dataset_prompts:
    prompt['merged_samples'] = list(
        map(
            lambda sample: apply_template(prompt, sample),
            tqdm(hf_exp_dataset['test']),
        )
    )
    prompt['original_samples'] = list(hf_exp_dataset['test'])

  0%|          | 0/10000 [00:00<?, ?it/s]

  0%|          | 0/10000 [00:00<?, ?it/s]

  0%|          | 0/10000 [00:00<?, ?it/s]

  0%|          | 0/10000 [00:00<?, ?it/s]

  0%|          | 0/10000 [00:00<?, ?it/s]

# Evaluate on each prompt and report the results

In [16]:
from datasets import DatasetDict
import re

def create_hf_dataset(dataset_prompt, columns=None):
  if columns is None:
    columns = ['text', 'summary']
  textes = []
  summaries = []
  for i,merged_sample in enumerate(dataset_prompt['merged_samples']):
    prefix= merged_sample.split('|||')[0]
    prefix = prefix.strip()
    prefix = prefix.replace('\xa0','')
    # prefix += '\nTranslation:'
    output = merged_sample.split('|||')[1].strip()
    textes.append(prefix)
    summaries.append(output)
  dataset = DatasetDict({ 'test' : datasets.Dataset.from_dict({
      columns[0]: textes,
      columns[1]: summaries,
  })})
  return dataset

In [17]:
dataset = create_hf_dataset(dataset_prompts[0])
dataset['test'][1],dataset['test'][1]

({'text': 'You are an expert Arabic text summarizer. The following article:\n"حذر زعيم الخضر الألماني جيم أوزدمير (معارضة) من منح تركيا حق استضافة بطولة كأس أوروبا لكرة القدم لعام 2024، مشيراًإلى وجود ""خطر كبير من استغلال نظام أردوغان (لهذا الحدث) لأغراض سياسية"". وذكر السياسي الألماني في حوار مع مجلة ""شبورتب يلد"" الرياضية الألمانية اليوم الأربعاء (20 سبتمبر/أيلول 2017)، أنه ""قلق جداً من أن تقوم دولة تحتجز رعايا ألمان كرهائن وتضعهم في السجن، لسبب وحيد يكمن في أنهم يمارسون عملهم كصحفيين أو كناشطين في مجال حقوق الانسان""، بتنظيم تظاهرة رياضية، فـ""كرة القدم لا يجب أن تصبح أداة في يد ديكتاتوريين ومستبدين"". وتأتي هذه التصريحات في أوج الحملة الانتخابية ياخذ ملف تركيا فيها حيزاً كبيراً بين المتنافسين. ومن المنتظر أن يحدد الاتحاد الأوروبي لكرة القدم (يويفا) في سبتمبر/ أيلول من العام القادم هوية البلد المنظم لكأس أوروبا 2024، مع العلم أن ألمانيا وتركيا مرشحتان لنيل شرف تنظيم البطولة. و.ب/ ع.غ (DW)"\ncan be summarized as:',
  'summary': '"زعيم حزب الخضر الألماني المنحدر من أصول تركية والمع

In [18]:
import torch
from lm_eval.models.vllm_causallms import VLLM
kwargs = dict(
    pretrained=MODEL_PATH,
    trust_remote_code=True,
    tensor_parallel_size=torch.cuda.device_count(),
    tokenizer=TOKENIZER_PATH,
    gpu_memory_utilization=0.9,
)

lm_obj = VLLM(**kwargs)

INFO 03-08 15:46:36 [utils.py:223] non-default args: {'tokenizer': '/raid_storage/shared_models/Qwen3-8B-Base', 'trust_remote_code': True, 'seed': 1234, 'disable_log_stats': True, 'model': '/raid_storage/shared_models/Qwen3-8B-Base'}


The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.
The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


INFO 03-08 15:46:36 [model.py:529] Resolved architecture: Qwen3ForCausalLM
INFO 03-08 15:46:36 [model.py:1549] Using max model len 32768
INFO 03-08 15:46:36 [scheduler.py:224] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 03-08 15:46:36 [vllm.py:689] Asynchronous scheduling is enabled.
(EngineCore_DP0 pid=1427612) INFO 03-08 15:46:36 [core.py:97] Initializing a V1 LLM engine (v0.16.0) with config: model='/raid_storage/shared_models/Qwen3-8B-Base', speculative_config=None, tokenizer='/raid_storage/shared_models/Qwen3-8B-Base', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat16, max_seq_len=32768, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=Structured

Loading safetensors checkpoint shards:   0% Completed | 0/5 [00:00<?, ?it/s]


(EngineCore_DP0 pid=1427612) INFO 03-08 15:46:45 [default_loader.py:293] Loading weights took 4.20 seconds
(EngineCore_DP0 pid=1427612) INFO 03-08 15:46:46 [gpu_model_runner.py:4221] Model loading took 15.27 GiB memory and 4.961832 seconds
(EngineCore_DP0 pid=1427612) INFO 03-08 15:46:54 [backends.py:916] Using cache directory: /raid_storage/SLURM/home/slurm_majedalshaibani/.cache/vllm/torch_compile_cache/018be9e9ea/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=1427612) INFO 03-08 15:46:54 [backends.py:976] Dynamo bytecode transform time: 7.98 s
(EngineCore_DP0 pid=1427612) INFO 03-08 15:47:01 [backends.py:267] Directly load the compiled graph(s) for compile range (1, 8192) from the cache, took 2.123 s
(EngineCore_DP0 pid=1427612) INFO 03-08 15:47:01 [monitor.py:34] torch.compile takes 10.10 s in total
(EngineCore_DP0 pid=1427612) INFO 03-08 15:47:02 [gpu_worker.py:373] Available KV cache memory: 54.57 GiB
(EngineCore_DP0 pid=1427612) INFO 03-08 15:47:02 [kv_cache_util

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 51/51 [00:02<00:00, 21.10it/s]
Capturing CUDA graphs (decode, FULL): 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 35/35 [00:01<00:00, 24.36it/s]


(EngineCore_DP0 pid=1427612) INFO 03-08 15:47:07 [gpu_model_runner.py:5246] Graph capturing finished in 5 secs, took 0.60 GiB
(EngineCore_DP0 pid=1427612) INFO 03-08 15:47:07 [core.py:278] init engine (profile, create kv cache, warmup model) took 20.87 seconds
INFO 03-08 15:47:08 [llm.py:355] Supported tasks: ['generate']


In [19]:
def evaluate_tasks(tasks,dataset_sub_path=TAWJEEH_DATASET_NAME):
    # MAKE SURE THE NOTEBOOK IS RUNNING FROM THE PROJECT ROOT!
    task_manager = TaskManager(include_path=f"eval_harness_extra_tasks/{dataset_sub_path}")
    results = simple_evaluate(  # call simple_evaluate
        model=lm_obj,
        tasks=tasks,
        num_fewshot=0,
        task_manager=task_manager,
    )
    return results

In [20]:
import json

def create_and_evaluate_single_prompt(prompt, save_results=True, force_re_evaluate=False):
    prompt_id = prompt['id']
    if TUNED_MODEL_PATH:
        results_dir = f'evaluation_results/{MODEL_NAME}-tuned/{TASK_NAME}/{TAWJEEH_DATASET_NAME}'
    else:
        results_dir = f'evaluation_results/{MODEL_NAME}/{TASK_NAME}/{TAWJEEH_DATASET_NAME}'
    prompt_results_file_path = f'{results_dir}/prompt_{prompt_id}.json'
    
    # Check if results exist and handle based on parameters
    if os.path.exists(prompt_results_file_path) and os.path.getsize(prompt_results_file_path) > 0:
        if not force_re_evaluate:
            print(f"Skipping prompt {prompt_id} - results already exist")
            with open(prompt_results_file_path, 'r') as f:
                prompt_results = json.load(f)
                print(make_table(prompt_results))
                return prompt_results
        else:
            print(f"Force re-evaluate enabled - reevaluating prompt {prompt_id}")
    
    # Create dataset and task files
    dataset = create_hf_dataset(prompt)
    
    # Save dataset
    dataset_dir = f'experimental_hf_datasets/{TAWJEEH_DATASET_NAME}/prompt_{prompt_id}'
    os.makedirs(dataset_dir, exist_ok=True)
    dataset['test'].to_parquet(f"{dataset_dir}/data.parquet")
    
    # Create YAML configuration
    yaml_text = f'''task: {TAWJEEH_DATASET_NAME}_prompt_{prompt_id}
dataset_path: experimental_hf_datasets/{TAWJEEH_DATASET_NAME}/prompt_{prompt_id}
output_type: generate_until
test_split: train
doc_to_text: text
doc_to_target: summary
metric_list:
  - metric: !function metrics.calculate_bleu
    aggregation: !function metrics.calculate_blue_agg
    higher_is_better: true
generation_kwargs:
    until:
    - <|eot_id|>
    - <|end_of_text|>
metadata:
  version: 1.0'''
    
    # Save YAML
    yaml_dir = f'eval_harness_extra_tasks/{TAWJEEH_DATASET_NAME}'
    os.makedirs(yaml_dir, exist_ok=True)
    with open(f'{yaml_dir}/prompt_{prompt_id}.yaml', 'w') as f:
        f.write(yaml_text)
    
    # Evaluate single prompt
    evaluation_task_name = f'{TAWJEEH_DATASET_NAME}_prompt_{prompt_id}'
    prompt_results = evaluate_tasks(tasks=[evaluation_task_name])
    
    print(make_table(prompt_results))
    
    # Save results if save_results is True
    if save_results:
        os.makedirs(results_dir, exist_ok=True)
        with open(prompt_results_file_path, 'w') as f:
            json.dump(prompt_results, f, ensure_ascii=False, indent=4, 
                     default=lambda o: '<not serializable>')
        print(f"Saved results for prompt {prompt_id}")
    else:
        print(f"Results not saved for prompt {prompt_id} (save_results=False)")
    
    print(f"Completed evaluation for prompt {prompt_id}")
    return prompt_results

In [21]:
def evaluate_all_prompts_sequentially(dataset_prompts, **kwargs):
    print(f"Starting sequential evaluation of {len(dataset_prompts)} prompts")
    all_results = {}
    
    for i, prompt in enumerate(dataset_prompts, 1):
        print('-' * 80)
        print(f"\nProcessing prompt {i}/{len(dataset_prompts)} (ID: {prompt['id']})")
        print("Template:", prompt['template'])
        print('-' * 80)
        
        prompt_results = create_and_evaluate_single_prompt(prompt, **kwargs)
        all_results[f"{TAWJEEH_DATASET_NAME}_prompt_{prompt['id']}"] = prompt_results
    
    return {'results': all_results}

In [22]:
all_results = evaluate_all_prompts_sequentially(dataset_prompts=dataset_prompts)

Starting sequential evaluation of 5 prompts
--------------------------------------------------------------------------------

Processing prompt 1/5 (ID: 14891)
Template: You are an expert Arabic text summarizer. The following article:
{{article}} 
can be summarized as:
|||
{{summary}}
--------------------------------------------------------------------------------
Skipping prompt 14891 - results already exist
|       Tasks       |Version|Filter|n-shot|    Metric    |   |Value |   |Stderr|
|-------------------|------:|------|-----:|--------------|---|-----:|---|------|
|AraSum_prompt_14891|      1|none  |     0|calculate_bleu|↑  |0.6648|±  |   N/A|

--------------------------------------------------------------------------------

Processing prompt 2/5 (ID: 14857)
Template: A summary of an article should be stand alone, providing a clear understanding of the article's content without requiring the reader to refer to the original text. It should focus on the most important information and

In [ ]:
exit()

ERROR 03-08 15:47:17 [core_client.py:616] Engine core proc EngineCore_DP0 died unexpectedly, shutting down client.


: 